# Phase 1 — Steering

The causal-intervention battery (steering, patching, noising, swap — Experiments 1-4 of the
original `vconf` pipeline) applied to three Benzon self-report/ground-truth pairs instead of
`confidence`/`correctness`:

1. **`benzon:synonyms` x `nuance_defined` x `nonbinary logit`** — the calibrated winner for the
   `nuance` construct on the synonyms dataset (`notebooks_benzon/phase_0_qwen/3_synonyms.ipynb`).
2. **List elicitation x `variety`** — `benzon_data.load_list_elicitation_items`, self-report
   `sentiment.VARIETY`, ground truth `activations.list_embedding_variety`.
3. **20 Questions x `impurity`** — `twenty_questions`, self-report `sentiment.IMPURITY`, ground
   truth `metrics.gini_impurity`.

This notebook does **steering** only (§4 of the original pipeline, `vconf/exp1_steering.py`) —
patching/noising/swap follow separately.

**Bug fixed to make this possible**: `exp1_steering.run_steering` computed
`intervention_metrics`' `confidence`-labeled columns using `interventions.numeric_midpoints`,
which returns `None` for every categorical prompt and silently falls back to
`metrics.MIDPOINTS` — a 10-class array hardcoded to `sentiment.CONFIDENCE`. That's only correct
by coincidence when `cfg.sentiment is CONFIDENCE`; for a 4-class sentiment like
`NUANCE_DEFINED`/`VARIETY`/`IMPURITY`, indexing `MIDPOINTS[0..3]` silently returns
*confidence's* midpoints instead of an `IndexError`. Fixed to derive midpoints from
`cfg.sentiment.class_midpoint` directly whenever the prompt is categorical. `logit_diff_change`/
`token_changed`/`clean_logit_diff`/`intervened_logit_diff` were never affected (they never read
midpoints) — only the `*_confidence` columns were silently wrong before this fix.

Runs under the default **reduced** profile (Qwen 2.5 7B) — see `nb.describe(cfg)`'s banner
below.


In [ ]:
import copy
import json as jsonlib
import pathlib
import random
import sys
from dataclasses import replace

import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

from vconf import activations
from vconf import exp1_steering as S1
from vconf import ground_truth as GT
from vconf import interventions as IV
from vconf import metrics as M
from vconf import notebook as nb
from vconf import data as datamod
from vconf import pipeline
from vconf import results as R
from vconf import twenty_questions as TQ
from vconf.prompts import LIST_ELICITATION_TEMPLATE, parse_list_items
from vconf.sentiment import CONFIDENCE, NUANCE_DEFINED, VARIETY, IMPURITY

base_cfg = nb.run_config("gemma-categorical", name="phase1-steering")
print(nb.describe(base_cfg))
# Gemma 3 27B in bf16 (~54 GB of weights alone) does not fit on one 48 GB GPU the way
# Qwen 7B does -- shard it across whatever's visible instead of the single-device default.
device_map = "auto" if nb.profile() == "paper" else None
loaded = nb.open_model(base_cfg, device_map=device_map)


## Shared steering runner

One function, reused for all three targets: given already-scored trials (`t.confidence`/
`t.class_index` populated by `pipeline.run_phase1`) and their rendered Phase-1 prompts, collect
PANL/PANL+1/CC/FCC activations, build the high-vs-low steering vector at each (layer, position),
pick a balanced test set, and run `exp1_steering.run_steering` across the full grid. `vector_n`
is scaled down from the paper's 25 for every Benzon target here — none of these datasets are
TriviaQA-scale, so demanding 25 clean high + 25 clean low trials the way §4.2 does would leave
some targets short of data; each target picks `vector_n` explicitly, sized to what it actually
has.

In [ ]:
from collections import Counter


def run_target_steering(
    loaded, cfg, trials, rendered, *, vector_n, test_n, require_correct, label,
):
    """Steering vectors from the whole trial set, tested on a balanced subset.

    Returns ``(frame, store, vectors)``, or ``(None, None, None)`` if the self-report turned
    out to have no variance at all to steer on (a real, reportable outcome for a small
    exploratory Benzon target, not something to force a vector out of).
    """
    layers = cfg.layers
    positions = cfg.positions
    class_counts = Counter(t.class_index for t in trials if t.class_index is not None)
    dist = {cfg.sentiment.classes[i]: n for i, n in sorted(class_counts.items())}
    print(f"[{label}] {len(trials)} trials, layers={layers}, positions={positions}")
    print(f"[{label}] self-report class distribution: {dist}")
    if len(class_counts) < 2:
        only = cfg.sentiment.classes[next(iter(class_counts))]
        print(f"[{label}] SKIPPED — self-report saturated to a single class ({only!r}); "
              f"no variance exists to build a steering vector from")
        return None, None, None

    store = activations.collect_activations(
        loaded, rendered, layers, positions,
        trial_ids=[t.qid for t in trials], batch_size=cfg.batch_size,
    )
    high_idx, low_idx = S1.select_vector_trials(trials, n=vector_n, require_correct=require_correct)
    print(f"[{label}] vector trials: {len(high_idx)} high / {len(low_idx)} low "
          f"(confidence range high={trials[high_idx[0]].confidence:.2f}..{trials[high_idx[-1]].confidence:.2f}, "
          f"low={trials[low_idx[0]].confidence:.2f}..{trials[low_idx[-1]].confidence:.2f})")
    vectors = S1.build_steering_vectors(store, high_idx, low_idx, scale_fraction=cfg.steering_scale_fraction)

    # Rank-based test split (top/bottom test_n//2 by raw confidence) instead of
    # select_test_trials' sentiment.high_band/low_band (the *named extreme* classes) --
    # those bands are badly underpopulated on these small Benzon self-report distributions.
    # On twenty_questions/impurity specifically, the band-based selection gave only 9 test
    # trials total, 8 of them "high" and just 1 "low" -- an almost-single-trial "low" result
    # that isn't a reliable read on anything. A rank-based split keeps the same balanced,
    # test_n-sized shape every other intervention in this pair of notebooks already uses
    # (exp2_patching's test pool, exp4_swap's recipient pools).
    order = np.argsort([t.confidence for t in trials])
    half = max(1, test_n // 2)
    test_idx = np.unique(np.concatenate([order[-half:], order[:half]]))
    test_trials, test_rendered = nb.subset(trials, rendered, test_idx)
    test_confidences = [t.confidence for t in test_trials]
    print(f"[{label}] {len(test_idx)} test trials (rank-based, confidence "
          f"{min(test_confidences):.2f}..{max(test_confidences):.2f})")

    frame = S1.run_steering(loaded, test_rendered, test_trials, vectors, cfg=cfg)
    frame["target"] = label
    return frame, store, vectors


def peak_summary(frame, label):
    """One row per (position, direction): the layer with the largest |confidence_change|."""
    if frame is None:
        return pd.DataFrame([{
            "target": label, "position": None, "direction": None, "peak_layer": None,
            "confidence_change": None, "logit_diff_change": None, "token_changed_rate": None,
        }]).iloc[0:0]
    summ = R.summarize(frame, by=("position", "direction", "layer"))
    rows = []
    for (position, direction), group in summ.groupby(["position", "direction"]):
        peak = group.loc[group["confidence_change_mean"].abs().idxmax()]
        rows.append({
            "target": label, "position": position, "direction": direction,
            "peak_layer": int(peak["layer"]),
            "confidence_change": round(float(peak["confidence_change_mean"]), 4),
            "logit_diff_change": round(float(peak["logit_diff_change_mean"]), 4),
            "token_changed_rate": round(float(peak["token_changed_mean"]), 3),
        })
    return pd.DataFrame(rows)


## Target A — `benzon:synonyms` x `nuance_defined`

120 (name1, name2, template) items. Ground truth for *which trials get steered* here is just
`nuance_defined`'s own self-report value — no correctness filter (`require_correct=False`):
"nuance" is about how multi-sided the answer reads, not about being right, and yes/no synonym
questions don't have a confidence-style notion of "the model was more sure when correct" the
way the original experiment's design assumed.

**Phase 0 stays `CONFIDENCE`, not `nuance_defined`** — same convention every calibration
notebook uses (`pipeline.run_multi_sentiment`'s `phase0_cfg = replace(base_cfg,
sentiment=CONFIDENCE)`): `build_phase0_prompt` bakes its `sentiment` argument's own instruction
block into the *answer-generation* prompt, so generating the Phase-0 answer under
`nuance_defined` framing would prime every answer toward a similarly-hedged style before
`nuance_defined` is ever asked about it as a Phase-1 follow-up — entangling the self-report with
its own priming rather than testing it against a neutral answer. `nuance_defined` only enters at
Phase 1.

In [ ]:
cfg_syn = nb.run_config(
    "gemma-categorical", dataset="benzon:synonyms", sentiment=NUANCE_DEFINED,
    ground_truth=GT.SynonymAnswerKey(), name="phase1-steering-synonyms",
)
print(nb.describe(cfg_syn))

items_syn = datamod.load_dataset_items(cfg_syn.dataset)
phase0_cfg_syn = replace(cfg_syn, sentiment=CONFIDENCE)
raw_syn = pipeline.run_phase0(loaded, items_syn, phase0_cfg_syn)
kept_syn, _ = pipeline.filter_positions_isolable(loaded, raw_syn, cfg_syn)
kept_syn = [t for t in kept_syn if t.valid]
rendered_kept_syn = pipeline.run_phase1(loaded, kept_syn, cfg_syn)
paired_syn = [(t, r) for t, r in zip(kept_syn, rendered_kept_syn) if t.class_index is not None]
trials_syn = [p[0] for p in paired_syn]
rendered_syn = [p[1] for p in paired_syn]
print(f"{len(trials_syn)}/{len(items_syn)} usable trials")


In [ ]:
frame_syn, store_syn, vectors_syn = run_target_steering(
    loaded, cfg_syn, trials_syn, rendered_syn,
    vector_n=20, test_n=24, require_correct=False, label="synonyms/nuance_defined",
)
summary_syn = peak_summary(frame_syn, "synonyms/nuance_defined")
display(summary_syn)


## Target B — 20 Questions x `impurity`

`notebooks_benzon/phase_0_qwen/5_twenty_questions.ipynb`'s own game loop
(`twenty_questions.py`), scaled down from its 10-games-per-category/10-turn design (1000 trials)
to keep this run's wall-clock time reasonable — each turn is two sequential generations (a
question, then a batched Yes/No partition over the remaining keywords), so game-play itself, not
the steering sweep, is the expensive part here. Every category is still represented; only the
per-category game count drops. Checkpointed to disk turn-by-turn so an interruption doesn't lose
already-played games.

In [ ]:
cfg_tq = nb.run_config("gemma-categorical", sentiment=IMPURITY, name="phase1-steering-twentyq")
print(nb.describe(cfg_tq))

N_GAMES_PER_CATEGORY = 1 if nb.profile() == "paper" else 2  # paper-profile Gemma is far slower per generate() call
N_TURNS = 10
UNIVERSE_SIZE = 100
CHECKPOINT_PATH = pathlib.Path(nb.TRIALS_DIR) / f"phase1-steering-twentyq-games-{cfg_tq.model_key}.json"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


def generate_text(loaded, cfg, prompt, max_new_tokens):
    messages = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    if cfg.use_chat_template:
        text = loaded.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = messages[-1]["content"]
    enc = loaded.tokenizer([text], return_tensors="pt", add_special_tokens=not cfg.use_chat_template).to(loaded.device)
    out = loaded.model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
        top_k=None, repetition_penalty=1.0, pad_token_id=loaded.tokenizer.pad_token_id,
    )
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    return loaded.tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()


def play_session(loaded, cfg, keywords, secret, n_turns):
    qa_history = []
    remaining = keywords
    records = []
    for i in range(n_turns):
        messages = TQ.build_conversation_for_next_question(keywords, qa_history)
        question = generate_text(loaded, cfg, messages, max_new_tokens=40)
        partition_text = generate_text(
            loaded, cfg, TQ.build_partition_prompt(question, remaining), max_new_tokens=600
        )
        yes_set, no_set = TQ.parse_partition(partition_text, remaining)
        answer = "Yes" if secret in yes_set else "No"
        remaining_after = yes_set if secret in yes_set else no_set
        records.append({
            "secret": secret, "turn": i, "qa_history_before": list(qa_history),
            "question": question, "answer": answer, "n_yes": len(yes_set), "n_no": len(no_set),
            "gini": M.gini_impurity(len(yes_set), len(no_set)),
        })
        qa_history.append((question, answer))
        remaining = remaining_after
    return qa_history, records


random.seed(0)
expected_turns = len(TQ.CATEGORIES) * N_GAMES_PER_CATEGORY * N_TURNS
if CHECKPOINT_PATH.exists():
    all_records = jsonlib.loads(CHECKPOINT_PATH.read_text())
else:
    all_records = []

if len(all_records) == expected_turns:
    print(f"[twentyq] loaded {len(all_records)} turns from checkpoint {CHECKPOINT_PATH} "
          f"(delete it to replay from scratch)")
else:
    all_records = []
    games = []
    for category in TQ.CATEGORIES:
        for game_in_category in range(N_GAMES_PER_CATEGORY):
            seed = len(games)
            game_keywords = TQ.sample_natural_keywords(n=UNIVERSE_SIZE, seed=seed, category=category)
            secret = random.choice(game_keywords)
            qa_history, records = play_session(loaded, cfg_tq, game_keywords, secret, N_TURNS)
            game_idx = len(games)
            for record in records:
                record["category"] = category
                record["game"] = game_idx
                record["keywords"] = list(game_keywords)
            games.append({"category": category, "game": game_idx, "keywords": list(game_keywords)})
            all_records.extend(records)
            CHECKPOINT_PATH.write_text(jsonlib.dumps(all_records, indent=1))
            print(f"[twentyq] played game {game_idx + 1}/{len(TQ.CATEGORIES) * N_GAMES_PER_CATEGORY} "
                  f"({category}) -> checkpointed {len(all_records)} turns")

turns_df = pd.DataFrame(all_records)
print(f"{len(all_records)} turns total")


In [ ]:
impurity_trials = []
for record in all_records:
    state = TQ.transcript_summary(tuple(record["keywords"]), record["qa_history_before"])
    qid = f"{record['category']}_game{record['game']}_turn{record['turn']}"
    impurity_trials.append(pipeline.Trial(qid=qid, question=state, answer=record["question"]))

impurity_rendered = pipeline.run_phase1(loaded, impurity_trials, cfg_tq)
turns_df["self_report_value"] = [t.confidence for t in impurity_trials]
turns_df["self_report"] = [IMPURITY.classes[t.class_index] for t in impurity_trials]
print(f"{len(impurity_trials)} impurity trials scored")
display(turns_df[["category", "game", "turn", "gini", "self_report", "self_report_value"]].head(10))


In [ ]:
frame_tq, store_tq, vectors_tq = run_target_steering(
    loaded, cfg_tq, impurity_trials, impurity_rendered,
    vector_n=25, test_n=24, require_correct=False, label="twenty_questions/impurity",
)
summary_tq = peak_summary(frame_tq, "twenty_questions/impurity")
display(summary_tq)


## Target C — List elicitation x `variety`

`notebooks_benzon/phase_0_qwen/4_list_elicitation.ipynb` ran this construct on exactly 5 trials
(one generated list per category) — nowhere near the ~2x`vector_n` clean high/low trials §4.2's
steering-vector methodology needs. Scaled up here by sampling `K_PER_CATEGORY` independent lists
per category (`do_sample=True`, temperature > 0) instead of one greedy generation each, keeping
the exact same construct (`sentiment.VARIETY`, `activations.list_embedding_variety`) — just more
draws of it, the same way the reduced profile scales *how much* runs, not *what* runs.

In [ ]:
cfg_le = nb.run_config(
    "gemma-categorical", dataset="benzon:list_elicitation", sentiment=VARIETY,
    name="phase1-steering-list-elicitation",
)
print(nb.describe(cfg_le))

import torch

K_PER_CATEGORY = 6 if nb.profile() == "paper" else 10  # paper-profile Gemma is far slower per generate() call
items_le = datamod.load_dataset_items(cfg_le.dataset)


def generate_list_sampled(loaded, cfg, prompt_text, seed, max_new_tokens=400):
    if cfg.use_chat_template:
        text = loaded.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}], tokenize=False, add_generation_prompt=True
        )
    else:
        text = prompt_text
    enc = loaded.tokenizer([text], return_tensors="pt", add_special_tokens=not cfg.use_chat_template).to(loaded.device)
    torch.manual_seed(seed)  # generate() takes no per-call generator kwarg; seed the global RNG instead
    out = loaded.model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.9, top_p=0.95,
        pad_token_id=loaded.tokenizer.pad_token_id,
    )
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    return loaded.tokenizer.decode(new_tokens[0], skip_special_tokens=True).strip()


list_trials = []
for item in items_le:
    for k in range(K_PER_CATEGORY):
        generated = generate_list_sampled(loaded, cfg_le, item.question, seed=hash((item.qid, k)) % (2**31))
        list_trials.append(pipeline.Trial(qid=f"{item.qid}_s{k}", question=item.question, answer=generated))
    print(f"[list_elicitation] {item.meta['category']}: {K_PER_CATEGORY} samples generated "
          f"(last one had {len(parse_list_items(list_trials[-1].answer))} parsed items)")

print(f"{len(list_trials)} generated lists")


In [ ]:
list_rendered = pipeline.run_phase1(loaded, list_trials, cfg_le)
# no correctness notion for "variety" (t.correct already defaults to None); steering ranks on
# self-report value alone (require_correct=False below)

variety_values = [t.confidence for t in list_trials]
print(f"variety self-report range: {min(variety_values):.2f}..{max(variety_values):.2f}, "
      f"distinct classes hit: {sorted(set(t.class_index for t in list_trials))}")


In [ ]:
frame_le, store_le, vectors_le = run_target_steering(
    loaded, cfg_le, list_trials, list_rendered,
    vector_n=10, test_n=16, require_correct=False, label="list_elicitation/variety",
)
summary_le = peak_summary(frame_le, "list_elicitation/variety")
display(summary_le)


## Combined summary

All three targets' peak layer/position deltas side by side — the same shape as
`exp1_steering.PAPER_TARGETS`, but for these three Benzon (sentiment, dataset) pairs instead of
confidence/TriviaQA. A real "PANL before CC" signature here would look like: PANL's peak layer
is earlier than CC's, and PANL's steering effect (`confidence_change`) is comparable in
magnitude to CC's rather than negligible next to it.

In [ ]:
combined = pd.concat([summary_syn, summary_tq, summary_le], ignore_index=True)
display(combined.sort_values(["target", "position", "direction"]))


In [ ]:
for target_name, group in combined.groupby("target"):
    panl = group[(group["position"] == "PANL")]
    cc = group[(group["position"] == "CC")]
    if len(panl) and len(cc):
        panl_layer = panl["peak_layer"].mean()
        cc_layer = cc["peak_layer"].mean()
        panl_effect = panl["confidence_change"].abs().mean()
        cc_effect = cc["confidence_change"].abs().mean()
        order = "PANL before CC" if panl_layer < cc_layer else "CC before/at PANL"
        print(f"{target_name}: PANL peak L{panl_layer:.1f} (|delta|={panl_effect:.3f})  "
              f"CC peak L{cc_layer:.1f} (|delta|={cc_effect:.3f})  -> {order}")
